# Trading bot

In [133]:
from ib_insync import *
import pandas as pd
import numpy as np
import datetime
import xgboost as xgb
from sklearn.metrics import accuracy_score, roc_curve, auc, roc_auc_score, precision_score, recall_score, confusion_matrix
import time
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
import ta
import plotly.graph_objects as go

util.startLoop()  # solo en Jupyter / entornos con event loop activo

## Fetch data

In [134]:
def fetch_ibkr_history(symbol_str, timeframe='1 hour', years=6):
    """
    Descarga histórico de IBKR en bloques para evitar violaciones de pacing.
    symbol_str: 'SPY' o 'SPXL'
    """
    ib = IB()
    try:
        ib.connect('127.0.0.1', 4002, clientId=13) # Ajusta puerto/ID según TWS
    except:
        print("No se pudo conectar a TWS. Asegúrate de que está abierto.")
        return None

    contract = Stock(symbol_str, 'SMART', 'USD')
    
    # Definimos fechas
    end_time = datetime.datetime.now()
    all_bars = []
    
    # Iteramos hacia atrás en bloques de 1 año (seguro para velas de 1h)
    for _ in range(years):
        print(f"Descargando bloque terminando en: {end_time}")
        
        bars = ib.reqHistoricalData(
            contract,
            endDateTime=end_time,
            durationStr='1 Y',
            barSizeSetting=timeframe,
            whatToShow='TRADES',
            useRTH=True,  # Regular Trading Hours (Importante para evitar ruido nocturno)
            formatDate=1
        )
        
        if not bars:
            break
            
        all_bars.extend(reversed(bars)) # Guardamos invertido para luego ordenar
        
        # Actualizamos la fecha de fin al comienzo del bloque actual
        end_time = bars[0].date
        time.sleep(2) # Pausa para respetar la API
        
    df = util.df(list(reversed(all_bars))) # Convertimos a DataFrame
    ib.disconnect()
    
    # Limpieza básica
    if df is not None:
        df['date'] = pd.to_datetime(df['date'])
        df.set_index('date', inplace=True)
        df = df[~df.index.duplicated(keep='first')] # Eliminar duplicados por solapamiento
    
    return df

# Ejemplo de uso:
# df_spy = fetch_ibkr_history('SPY', years=10)
df_spxl = fetch_ibkr_history('SPXL', years=10)

Descargando bloque terminando en: 2026-02-07 17:29:58.415821
Descargando bloque terminando en: 2025-02-07 09:30:00-05:00
Descargando bloque terminando en: 2024-02-08 09:30:00-05:00
Descargando bloque terminando en: 2023-02-08 09:30:00-05:00
Descargando bloque terminando en: 2022-02-08 09:30:00-05:00
Descargando bloque terminando en: 2021-02-08 09:30:00-05:00
Descargando bloque terminando en: 2020-02-07 09:30:00-05:00
Descargando bloque terminando en: 2019-02-07 09:30:00-05:00
Descargando bloque terminando en: 2018-02-07 09:30:00-05:00
Descargando bloque terminando en: 2017-02-07 09:30:00-05:00


## Features & Target

In [136]:
import pandas as pd
import numpy as np
from ta.momentum import RSIIndicator
from ta.trend import SMAIndicator, ADXIndicator
from ta.volatility import BollingerBands, AverageTrueRange

def preparar_dataset_con_filtros(df_raw, profit_thr=0.01, stop_thr=0.02):
    """
    Genera dataset filtrando días tóxicos (Alta Volatilidad o Ruido).
    """
    df = df_raw.copy().sort_index()

    # 1. ARREGLAR FECHAS
    if isinstance(df.index, pd.DatetimeIndex):
        if df.index.tz is not None:
             df.index = df.index.tz_convert('US/Eastern').tz_localize(None)
        df['Day'] = df.index.astype(str).str[:10]

    # 2. INDICADORES (Añadimos ATR y ADX para los filtros)
    # ATR: Mide el tamaño de las velas (Volatilidad)
    atr_ind = AverageTrueRange(high=df['high'], low=df['low'], close=df['close'], window=14)
    df['atr'] = atr_ind.average_true_range()
    df['atr_pct'] = df['atr'] / df['close'] # ATR relativo (ej: 0.015 = 1.5%)

    # ADX: Mide la fuerza de la tendencia (0-100). <20 es ruido.
    adx_ind = ADXIndicator(high=df['high'], low=df['low'], close=df['close'], window=14)
    df['adx'] = adx_ind.adx()

    # Otros indicadores estándar
    rsi_ind = RSIIndicator(close=df['close'], window=14)
    df['rsi'] = rsi_ind.rsi()
    sma_ind = SMAIndicator(close=df['close'], window=20)
    df['sma_20'] = sma_ind.sma_indicator()
    bb_ind = BollingerBands(close=df['close'], window=20, window_dev=2)
    df['bb_width'] = bb_ind.bollinger_wband()

    data_list = []

    # 3. LOOP POR DÍAS
    for date_str, day_df in df.groupby('Day'):
        
        # Filtro básico de datos mínimos
        if len(day_df) < 5: continue
        
        try:
            # Vela de decisión (12:00 PM aprox, índice 2)
            vela_decision = day_df.iloc[2] 
        except IndexError:
            continue

        # ============================================================
        # 🛑 ZONA DE REGLAS DURAS (HARD FILTERS)
        # ============================================================
        
        # REGLA 1: FILTRO DE PÁNICO (Volatilidad Excesiva)
        # Si el ATR diario supera el 2.5%, el mercado está muy peligroso.
        # En SPXL (3x), un ATR del 2.5% en el subyacente implica movimientos del 7.5%.
        # Es casi imposible poner un Stop Loss sensato aquí.
        if vela_decision['atr_pct'] > 0.025: 
            # print(f"Día {date_str} descartado por volatilidad extrema ({vela_decision['atr_pct']:.2%})")
            continue

        # REGLA 2: FILTRO DE RUIDO (ADX Muerto)
        # Si el ADX está por debajo de 20, el mercado está en rango lateral.
        # Los sistemas de ruptura (breakout) fallan mucho aquí.
        if vela_decision['adx'] < 20:
            # print(f"Día {date_str} descartado por falta de tendencia (ADX < 20)")
            continue
            
        # REGLA 3: FILTRO DE APERTURA ABURRIDA (Opcional)
        # Si el volumen de las 3 primeras horas es ridículo, no entres.
        vol_promedio = df['volume'].rolling(50).mean().iloc[-1] # Aprox
        vol_hoy_acum = day_df.iloc[0:3]['volume'].sum()
        # Si no hay volumen, suele ser día festivo parcial o espera de noticias (Fed)
        if vol_hoy_acum < (vol_promedio * 0.1): # Ajustar según tus datos
             continue

        # ============================================================
        # ✅ FIN DE FILTROS - SI LLEGA AQUÍ, EL DÍA ES "OPERABLE"
        # ============================================================

        precio_entrada = vela_decision['close']
        precio_tp = precio_entrada * (1 + profit_thr)
        precio_sl = precio_entrada * (1 - stop_thr)

        if np.isnan(vela_decision['rsi']): continue

        # FEATURES (X)
        row = {
            'Date': pd.to_datetime(date_str),
            'rsi_val': vela_decision['rsi'],
            'adx_val': vela_decision['adx'], # Añadimos ADX como feature también
            'atr_pct': vela_decision['atr_pct'],
            'dist_sma_20': (precio_entrada - vela_decision['sma_20']) / vela_decision['sma_20'],
            'volatility_width': vela_decision['bb_width'],
            'return_open_to_decision': (precio_entrada - day_df.iloc[0]['open']) / day_df.iloc[0]['open'],
        }

        # TARGET (y)
        velas_futuras = day_df.iloc[3:]
        resultado_operacion = 0 
        trade_cerrado = False
        
        if len(velas_futuras) > 0:
            for i, vela in velas_futuras.iterrows():
                if vela['low'] <= precio_sl:
                    resultado_operacion = 0 
                    trade_cerrado = True
                    break 
                if vela['high'] >= precio_tp:
                    resultado_operacion = 1 
                    trade_cerrado = True
                    break 

            if not trade_cerrado:
                # Cierre por tiempo
                if day_df.iloc[-1]['close'] > precio_entrada:
                    resultado_operacion = 1
                else:
                    resultado_operacion = 0
        else:
            continue
        
        row['target'] = resultado_operacion
        data_list.append(row)

    if not data_list:
        return pd.DataFrame()

    return pd.DataFrame(data_list).set_index('Date')

## Training

In [137]:
def train_model(df):
    X = df.drop(columns='target')
    y = df['target']
    
    # Split temporal simple (último 20% para test)
    split_point = int(len(df) * 0.8)
    X_train, X_test = X.iloc[:split_point], X.iloc[split_point:]
    y_train, y_test = y.iloc[:split_point], y.iloc[split_point:]
    
    # Configuración del modelo (Robusta para evitar overfitting)
    model = xgb.XGBClassifier(
    n_estimators=100,        # Más árboles
    max_depth=3,             # Árboles MUY pequeños (evita memorizar ruido)
    learning_rate=0.05,      # Aprendizaje lento
    subsample=0.5,           # Usa solo la mitad de los datos por árbol (reduce varianza)
    colsample_bytree=0.5,    # Usa solo la mitad de features por árbol
    gamma=0.5,               # <--- CLAVE: Umbral mínimo para crear una rama
    reg_alpha=0.1,           # Regularización L1 (Lasso)
    reg_lambda=1.0,          # Regularización L2 (Ridge)
    scale_pos_weight=1,      # Ajustar según ratio (ej: Negativos / Positivos)
    eval_metric='auc'
    )
    
    model.fit(X_train, y_train)
    
    # Predicciones
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1] # Probabilidad de clase 1
    
    print(f"Precision Score (Test): {precision_score(y_test, preds):.2f}")
    
    return model, X_test, y_test, probs

In [ ]:
# Features and target
df_spxl_features_and_target = preparar_dataset_con_filtros(df_spxl, profit_thr=0.015)

# Train model
model, X_test, y_test, y_test_probs = train_model(df_spxl_features_and_target)

Precision Score (Test): 0.54


## Model evaluation

In [139]:
MODEL_THRESHOLD = 0.60

def evaluate_trading_model(model, X_test, y_test, prob_threshold=0.6):
    """
    Evalúa el modelo con un umbral de decisión específico.
    Retorna un diccionario con las métricas clave.
    """
    
    # 1. Obtener probabilidades (Clase 1 = Compra)
    probs = model.predict_proba(X_test)[:, 1]
    
    # 2. Calcular AUC (Independiente del umbral, mide la calidad global)
    try:
        auc = roc_auc_score(y_test, probs)
    except:
        auc = 0.5 # Si falla (ej: solo hay una clase en test), asumimos random
        
    # 3. Aplicar el Umbral (Convertir probabilidad en señal 0 o 1)
    # Importante: No usamos model.predict() porque eso usa umbral 0.5 por defecto
    predictions = (probs > prob_threshold).astype(int)
    
    # 4. Métricas de Trading
    n_signals = predictions.sum()
    total_samples = len(y_test)
    
    if n_signals > 0:
        win_rate = precision_score(y_test, predictions) # % de veces que acertamos al comprar
    else:
        win_rate = 0.0
        
    # 5. Desglose de Aciertos/Fallos (Confusion Matrix)
    # TN: No compramos y bajó (Bien) | FP: Compramos y bajó (MAL - Pérdida)
    # FN: No compramos y subió (Oportunidad perdida) | TP: Compramos y subió (BIEN - Ganancia)
    tn, fp, fn, tp = confusion_matrix(y_test, predictions).ravel()
    
    # --- REPORTE IMPRESO ---
    print(f"========== EVALUACIÓN MODELO (Umbral > {prob_threshold}) ==========")
    print(f"1. CALIDAD GLOBAL (AUC):      {auc:.2%} {'(Bueno)' if auc > 0.55 else '(Ruido)'}")
    print(f"2. ACTIVIDAD:")
    print(f"   - Señales Emitidas:        {n_signals} de {total_samples} velas ({n_signals/total_samples:.1%})")
    print(f"3. RENDIMIENTO (Win Rate):    {win_rate:.2%}  <-- DATO CLAVE")
    print(f"4. DETALLE OPERATIVO:")
    print(f"   - ✅ Aciertos (TP):        {tp}")
    print(f"   - ❌ Fallos (FP):          {fp} (Stop Loss)")
    print(f"   - 💤 Oportunidades (FN):   {fn} (Señales que perdimos)")
    print("===============================================================")
    
    return {
        'AUC': auc,
        'Signals': n_signals,
        'Win_Rate': win_rate,
        'TP': tp, 
        'FP': fp,
        'Probs': probs # Devolvemos las probs por si quieres graficar luego
    }

# --- EJEMPLO DE USO ---
metrics = evaluate_trading_model(model, X_test, y_test, prob_threshold=MODEL_THRESHOLD)

========== EVALUACIÓN MODELO (Umbral > 0.6) ==========
1. CALIDAD GLOBAL (AUC):      53.19% (Ruido)
2. ACTIVIDAD:
   - Señales Emitidas:        87 de 288 velas (30.2%)
3. RENDIMIENTO (Win Rate):    59.77%  <-- DATO CLAVE
4. DETALLE OPERATIVO:
   - ✅ Aciertos (TP):        52
   - ❌ Fallos (FP):          35 (Stop Loss)
   - 💤 Oportunidades (FN):   104 (Señales que perdimos)


In [140]:
def plot_model_performance_v2(df_hourly, X_test, y_test, probs, threshold=0.6):
    """
    Versión robusta: Fuerza la conversión a DatetimeIndex para evitar AttributeError.
    """
    
    # --- 1. PREPARACIÓN Y NORMALIZACIÓN DE DATOS ---
    df_plot = df_hourly.copy()
    
    # Unificamos X_test, y_test y Probs
    results_df = X_test.copy()
    
    # Aseguramos que 'target' esté presente
    if isinstance(y_test, pd.Series):
        results_df['target'] = y_test.values
    elif isinstance(y_test, pd.DataFrame):
        results_df['target'] = y_test.iloc[:, 0].values # Toma la primera columna si es DF
    else:
        results_df['target'] = y_test # Si es array numpy
        
    results_df['Prob'] = probs
    
    # --- 2. CORRECCIÓN DE FECHAS (EL FIX) ---
    
    # A) Datos Horarios (Fondo del gráfico)
    # PASO CRÍTICO: Forzamos que sea DatetimeIndex. Si ya lo es, no hace nada. Si es texto, lo convierte.
    df_plot.index = pd.to_datetime(df_plot.index)
    
    # Ahora que sabemos seguro que es fecha, miramos la zona horaria
    if df_plot.index.tz is not None:
        # Convertimos a NY y quitamos la zona para comparar limpio
        df_plot.index = df_plot.index.tz_convert('US/Eastern').tz_localize(None)
    
    # B) Datos Diarios (Resultados del Test)
    results_df.index = pd.to_datetime(results_df.index)
    
    if results_df.index.tz is not None:
        results_df.index = results_df.index.tz_localize(None)

    # --- 3. FILTRADO DE OPERACIONES ---
    trades = results_df[results_df['Prob'] > threshold].copy()
    
    if trades.empty:
        print(f"⚠️ El modelo no realizó ninguna operación con umbral > {threshold}.")
        return

    # --- 4. MAPEO EXACTO A VELAS ---
    timestamps = []
    prices = []
    colors = []
    hover_texts = []
    
    # Asumimos entrada a las 12:00 PM
    delta_entrada = pd.Timedelta(hours=12, minutes=0)

    print(f"Procesando {len(trades)} operaciones...")

    for date, row in trades.iterrows():
        try:
            # Calculamos hora exacta
            entry_time = date + delta_entrada
            
            # Buscamos vela más cercana
            idx_loc = df_plot.index.get_indexer([entry_time], method='nearest')[0]
            candle = df_plot.iloc[idx_loc]
            
            # Validar que no sea un día festivo (vela muy lejana)
            if abs(candle.name - entry_time) > pd.Timedelta(hours=2):
                continue

            timestamps.append(candle.name)
            prices.append(candle['close'])
            
            # Color según Target
            is_win = row['target'] == 1
            if is_win:
                colors.append('#00FF00') # Verde
                msg = "WIN"
            else:
                colors.append('#FF0000') # Rojo
                msg = "LOSS"
                
            hover_texts.append(f"Fecha: {date.date()}<br>Prob: {row['Prob']:.1%}<br>{msg}")

        except Exception as e:
            continue

    # --- 5. VISUALIZACIÓN ---
    # Zoom automático
    if len(results_df) > 0:
        start_dt = results_df.index[0] - pd.Timedelta(days=5)
        end_dt = results_df.index[-1] + pd.Timedelta(days=5)
        df_zoom = df_plot.loc[(df_plot.index >= start_dt) & (df_plot.index <= end_dt)]
    else:
        df_zoom = df_plot.iloc[-100:] # Fallback si algo falla

    fig = go.Figure()

    # Velas
    fig.add_trace(go.Candlestick(
        x=df_zoom.index,
        open=df_zoom['open'], high=df_zoom['high'],
        low=df_zoom['low'], close=df_zoom['close'],
        name='SPXL',
        increasing_line_color='#26a69a', 
        decreasing_line_color='#ef5350'
    ))

    # Señales
    fig.add_trace(go.Scatter(
        x=timestamps, y=prices,
        mode='markers',
        marker=dict(color=colors, size=14, symbol='triangle-up', line=dict(width=1, color='black')),
        text=hover_texts, hoverinfo='text',
        name='Señal'
    ))

    fig.update_layout(
        title=f'Backtest Visual (Umbral > {threshold:.2f})',
        yaxis_title='Precio USD',
        template='plotly_dark',
        height=800,
        xaxis_rangeslider_visible=False,
        xaxis=dict(
            rangebreaks=[
                dict(bounds=["sat", "mon"]), 
                dict(bounds=[16, 9.5], pattern="hour"), 
            ]
        )
    )

    fig.show()

plot_model_performance_v2(df_spxl, X_test, y_test, y_test_probs, MODEL_THRESHOLD)

Procesando 87 operaciones...


In [141]:
def simulate_equity_curve(y_test, probs, threshold=0.60, 
                          tp_pct=0.01, sl_pct=0.02, 
                          comm_pct=0.001): # 0.1% de comisión+slippage (conservador)
    """
    Simula el crecimiento de la cuenta basado en las señales del modelo.
    """
    
    # 1. Preparar datos
    df_sim = pd.DataFrame({'Target': y_test, 'Prob': probs})
    
    # 2. Filtrar solo las operaciones ejecutadas
    trades = df_sim[df_sim['Prob'] > threshold].copy()
    
    capital = 10000.0 # Capital Inicial
    equity_curve = [capital]
    dates = [] # Si tuvieras fechas, úsalas aquí
    
    wins = 0
    losses = 0
    
    print(f"--- SIMULACIÓN DE CARTERA (Umbral > {threshold}) ---")
    print(f"TP: {tp_pct*100}% | SL: {sl_pct*100}% | Costes: {comm_pct*100}%")
    
    for i, row in trades.iterrows():
        # Resultado bruto
        if row['Target'] == 1:
            pnl_pct = tp_pct
            wins += 1
        else:
            pnl_pct = -sl_pct
            losses += 1
            
        # Restamos comisiones (se aplican al entrar y al salir, simplificado como coste total)
        pnl_neto = pnl_pct - comm_pct
        
        # Interés Compuesto
        capital = capital * (1 + pnl_neto)
        equity_curve.append(capital)
    
    # Resultados Finales
    total_return = (capital - 10000) / 10000
    dd_curve = pd.Series(equity_curve)
    max_dd = (dd_curve / dd_curve.cummax() - 1).min()
    
    print(f"Capital Final:   ${capital:,.2f}")
    print(f"Retorno Total:   {total_return:.2%}")
    print(f"Max Drawdown:    {max_dd:.2%}")
    print(f"Ratio Win/Loss:  {wins}/{losses}")

    # Gráfico
    fig = go.Figure()
    fig.add_trace(go.Scatter(y=equity_curve, mode='lines', name='Capital', fill='tozeroy'))
    fig.update_layout(title='Curva de Capital Simulada', yaxis_title='USD ($)', template='plotly_dark')
    fig.show()

# --- EJECUCIÓN ---
# Prueba con diferentes umbrales para ver cuál maximiza el dinero, no solo el acierto
simulate_equity_curve(y_test, y_test_probs, threshold=0.65, tp_pct=0.015, sl_pct=0.015)

--- SIMULACIÓN DE CARTERA (Umbral > 0.65) ---
TP: 1.5% | SL: 1.5% | Costes: 0.1%
Capital Final:   $12,027.33
Retorno Total:   20.27%
Max Drawdown:    -7.49%
Ratio Win/Loss:  33/17


## BUY / SELL strategy

In [142]:
def get_actionable_signal(model, current_data_row, probability_threshold=0.60):
    """
    Retorna la decisión basada en la probabilidad del modelo.
    threshold: 0.60 (Exigimos un 60% de certeza, no un 50%)
    """
    # Asumimos que current_data_row ya tiene las features calculadas
    features_val = current_data_row.values.reshape(1, -1)
    
    # Predecir probabilidad
    prob_up = model.predict_proba(features_val)[0][1]
    
    decision = "NEUTRAL"
    if prob_up > probability_threshold:
        decision = "COMPRA FUERTE (BULL)"
    elif prob_up < 0.4:
        decision = "EVITAR / VENTA"
        
    return decision, prob_up